# Paso 4.2 — LSTM para Series Temporales

## Desafío Profesional: Cambio Climático y Energías Renovables en Argentina
### Etapa 4 — Redes Neuronales (Paso 4.2)

---

## 🎯 Objetivo del notebook

Aplicar redes neuronales recurrentes tipo **LSTM** para pronosticar la generación mensual de energía renovable en Argentina (2018), y compararlas contra el baseline **ARIMA(1,1,1)** obtenido en la Etapa 3.

## 📍 ¿Por qué LSTM aquí?

En el Paso 4.1 vimos que las redes neuronales **densas** (MLP) no le ganan a Ridge en un problema de regresión estático. El feature engineering manual es competitivo.

Pero las series temporales son un mundo distinto:

- Los datos tienen un **orden** que importa: el valor de enero depende del de diciembre, no al revés.
- Hay **patrones de memoria**: lo que pasó hace 12 meses (un ciclo anual) suele repetirse.
- Las relaciones pueden ser **no lineales**: un cambio estructural (como la conexión de parques eólicos nuevos en 2018) no es replicable con una simple tendencia lineal.

**Aquí la LSTM tiene una ventaja estructural genuina**: fue diseñada exactamente para aprender dependencias secuenciales de largo plazo. Si Deep Learning va a justificarse en algún lado de este proyecto, es acá.

## 🆚 El rival a batir

| Modelo | MAE | RMSE | MAPE |
|---|---|---|---|
| **ARIMA(1,1,1)** | 81.8 GWh | 105.1 GWh | **26.7%** |

ARIMA captura bien la tendencia, pero **subestima el crecimiento acelerado de 2018** (cuando entraron en operación varios parques eólicos grandes de la ronda RenovAr). Ese es justamente el escenario donde la no-linealidad de una LSTM puede marcar diferencia.

## 🗺️ Plan del notebook

1. Setup y carga de datos
2. Exploración rápida de la serie
3. Reproducir baseline ARIMA (para tenerlo en el mismo notebook)
4. Preparación de datos para LSTM (ventanas, escalado, split temporal)
5. **LSTM Simple** (univariada, 1 capa)
6. **LSTM Multivariada** (agrega demanda eléctrica + encoding cíclico del mes)
7. Comparación final y storytelling

> 💡 **Decisión de diseño:** nos quedamos con 2 arquitecturas LSTM (no 3). Con solo 84 meses de entrenamiento, modelos más profundos tienden a sobreajustar. La pregunta interesante no es _"¿cuán grande hago la red?"_ sino _"¿agregar contexto externo ayuda o la serie sola alcanza?"_.


## 1. Setup: imports, semillas y carga de datos

Antes de cualquier cosa, fijamos las semillas aleatorias. Las LSTMs son especialmente sensibles a la inicialización de pesos, y sin esto cada ejecución daría resultados distintos. Fijamos Python, NumPy y TensorFlow.


In [ ]:
# Imports estándar
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modelos
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Reproducibilidad
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paleta de colores del proyecto
COLORS = {
    'primary': '#1B4F72',   # Azul oscuro — datos reales / test
    'accent':  '#2E86C1',   # Azul claro — datos de train
    'success': '#27AE60',   # Verde — LSTM Simple
    'warning': '#F39C12',   # Naranja — LSTM Multivariada
    'danger':  '#E74C3C',   # Rojo — (reservado)
    'gray':    '#95A5A6',   # Gris — ARIMA
}

# Estilo de gráficos
plt.rcParams['figure.dpi'] = 90
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print(f"✅ TensorFlow {tf.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}")
print(f"✅ Semilla fija: {SEED}")

### Carga del dataset mensual

> 📁 **Nota para Colab:** ajustá la ruta a la carpeta donde tengas guardados tus datasets limpios. En el notebook original del desafío, la ruta es `/content/drive/MyDrive/desafio_profesional/datos_limpios/`.


In [ ]:
# Ruta del dataset — ajustar si se ejecuta en Colab
# RUTA = '/content/drive/MyDrive/desafio_profesional/datos_limpios/dataset_argentina_mensual.csv'
RUTA = 'dataset_argentina_mensual.csv'

df = pd.read_csv(RUTA, parse_dates=['mes'])
df = df.sort_values('mes').reset_index(drop=True)

print(f"📊 Shape: {df.shape}")
print(f"📅 Rango: {df['mes'].min().strftime('%Y-%m')} a {df['mes'].max().strftime('%Y-%m')}")
print(f"📐 Columnas disponibles: {list(df.columns)}")
df.head()

## 2. Exploración rápida de la serie temporal

Antes de modelar, necesitamos **ver** la serie. Dos cosas saltan a la vista en estos datos:

1. **Tendencia creciente**: Argentina aumentó su generación renovable progresivamente desde 2011.
2. **Punto de inflexión en 2018**: la pendiente se empina fuerte porque se conectaron parques eólicos nuevos (el famoso "boom" de RenovAr).

Este cambio estructural es clave: nuestro modelo de train (2011-2017) **no vio nunca** el régimen de 2018. Va a tener que **extrapolar**.


In [ ]:
# Preparar serie con índice temporal
ts = df.set_index('mes')['gen_renovable_total_GWh'].asfreq('MS')

# Split temporal — mismo que ARIMA en Etapa 3
SPLIT_DATE = '2018-01-01'
ts_train = ts[ts.index < SPLIT_DATE]
ts_test  = ts[ts.index >= SPLIT_DATE]

print(f"Train: {len(ts_train)} meses ({ts_train.index.min().strftime('%Y-%m')} a {ts_train.index.max().strftime('%Y-%m')})")
print(f"Test:  {len(ts_test)} meses ({ts_test.index.min().strftime('%Y-%m')} a {ts_test.index.max().strftime('%Y-%m')})")
print(f"\nEstadísticas train: media={ts_train.mean():.1f} GWh, máx={ts_train.max():.1f} GWh")
print(f"Estadísticas test:  media={ts_test.mean():.1f} GWh, máx={ts_test.max():.1f} GWh")
print(f"\n⚠️  El máximo de test ({ts_test.max():.0f} GWh) supera al máximo de train ({ts_train.max():.0f} GWh).")
print(f"   Cualquier modelo va a tener que EXTRAPOLAR para capturar el crecimiento de 2018.")

In [ ]:
# Visualización de la serie
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ts_train.index, ts_train.values, color=COLORS['accent'],
        linewidth=1.8, label='Train (2011-2017)')
ax.plot(ts_test.index, ts_test.values, color=COLORS['primary'],
        linewidth=2.2, label='Test (2018)')

# Línea vertical del split
ax.axvline(pd.Timestamp(SPLIT_DATE), color=COLORS['gray'],
           linestyle='--', alpha=0.6, label='Split train/test')

# Anotación del cambio estructural
ax.annotate('Boom RenovAr:\nparques eólicos\nnuevos', xy=(pd.Timestamp('2018-09-01'), 400),
            xytext=(pd.Timestamp('2016-06-01'), 420),
            fontsize=10, color=COLORS['primary'],
            arrowprops=dict(arrowstyle='->', color=COLORS['gray'], alpha=0.7))

ax.set_title('Generación renovable mensual en Argentina (2011-2018)', fontsize=13, weight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Generación (GWh)')
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

print(f"\n📈 La serie tiene TENDENCIA (sube), ESTACIONALIDAD (oscila anualmente) y un CAMBIO ESTRUCTURAL en 2018.")

## 3. Baseline ARIMA — reproducción

Ya tenemos los resultados de ARIMA de la Etapa 3, pero los **reproducimos acá** para tener todas las métricas en el mismo notebook y poder comparar fácilmente.

**¿Qué es ARIMA(1,1,1)?** Descomponemos el nombre:
- **AR(1)**: el valor de hoy depende del valor de ayer (1 lag autoregresivo).
- **I(1)**: le aplicamos una diferenciación para quitar la tendencia (el modelo trabaja con _cambios_ mes a mes, no con _niveles_).
- **MA(1)**: el error de hoy depende del error de ayer (1 término de media móvil).

Es un modelo **lineal** y **paramétrico**: con 3 parámetros intenta capturar la dinámica de la serie. Simple, interpretable, y sorprendentemente competitivo.


In [ ]:
# Ajustar ARIMA(1,1,1)
arima_model = ARIMA(ts_train, order=(1, 1, 1))
arima_fit = arima_model.fit()

# Pronóstico de 12 meses
arima_forecast = arima_fit.forecast(steps=len(ts_test))
arima_forecast.index = ts_test.index

# Métricas
mae_arima  = mean_absolute_error(ts_test, arima_forecast)
rmse_arima = np.sqrt(mean_squared_error(ts_test, arima_forecast))
mape_arima = np.mean(np.abs((ts_test.values - arima_forecast.values) / ts_test.values)) * 100

print(f"📊 ARIMA(1,1,1):")
print(f"   MAE  = {mae_arima:.2f} GWh")
print(f"   RMSE = {rmse_arima:.2f} GWh")
print(f"   MAPE = {mape_arima:.2f}%")

In [ ]:
# Visualización del forecast ARIMA
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ts_train.index, ts_train.values, color=COLORS['accent'],
        linewidth=1.5, label='Train', alpha=0.7)
ax.plot(ts_test.index, ts_test.values, color=COLORS['primary'],
        linewidth=2.5, label='Real 2018')
ax.plot(arima_forecast.index, arima_forecast.values, color=COLORS['gray'],
        linewidth=2.2, linestyle='--', marker='o', markersize=5,
        label=f'ARIMA(1,1,1) — MAPE {mape_arima:.1f}%')

ax.axvline(pd.Timestamp(SPLIT_DATE), color=COLORS['gray'], linestyle=':', alpha=0.5)
ax.set_title('Baseline ARIMA(1,1,1) — Forecast 2018', fontsize=13, weight='bold')
ax.set_xlabel('Mes'); ax.set_ylabel('GWh'); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

print(f"\n🔍 Observación: el forecast de ARIMA es casi PLANO en 2018.")
print(f"   La razón: al diferenciar la serie, ARIMA asume que el cambio mensual futuro se parece al pasado.")
print(f"   Pero en 2018 el cambio mensual fue MUCHO mayor que en cualquier otro año de la historia.")
print(f"   Por eso subestima tanto: no puede 'inventar' una aceleración que nunca vio.")

## 4. Preparación de datos para LSTM

Aquí es donde todo se pone interesante. Las LSTMs necesitan los datos en un formato muy específico, y si nos equivocamos acá, todo lo demás no importa.

### 4.1 El concepto de **ventana deslizante**

Una LSTM no ve la serie completa de golpe. Ve **fragmentos** (ventanas) de longitud fija.

Pensalo así: si mirás por la ventanilla de un auto en movimiento, siempre ves los _últimos N metros_ de camino. Eso es la ventana. A medida que el auto avanza, la ventana se desliza. Cada "foto" de la ventana es un ejemplo de entrenamiento.

Con una ventana de **12 meses** (`n_lags=12`), tenemos:

```
Ejemplo 1: [ene-2011, feb-2011, ..., dic-2011]  →  predecir  ene-2012
Ejemplo 2: [feb-2011, mar-2011, ..., ene-2012]  →  predecir  feb-2012
Ejemplo 3: [mar-2011, abr-2011, ..., feb-2012]  →  predecir  mar-2012
...
```

**¿Por qué 12?** Porque capturamos un **ciclo anual completo**. La generación hidroeléctrica sube en invierno (julio-agosto) y la solar en verano (enero-febrero). Si el modelo ve 12 meses hacia atrás, puede aprender ese patrón.

Con 84 meses de train y `n_lags=12`, generamos **72 ejemplos de entrenamiento**. Es poco, pero suficiente si mantenemos la arquitectura simple.

### 4.2 Por qué escalamos SOLO con el train

Las redes neuronales aprenden mejor cuando los valores están en rangos cercanos a 0. Usamos `StandardScaler` (centra en 0, varianza 1).

**Regla de oro:** el scaler se ajusta (`.fit`) **solo con el train**. Si usáramos los datos de test para ajustar el scaler, estaríamos filtrando información del futuro (data leakage). Siempre tratamos al test como si fuera "desconocido".

### 4.3 La forma 3D que espera la LSTM

Keras pide un tensor con shape `(samples, timesteps, features)`:

- **samples** = cantidad de ventanas
- **timesteps** = largo de la ventana (12)
- **features** = variables por timestep (1 para univariado, más para multivariado)


In [ ]:
def create_sequences(data, n_lags, target_col=0):
    """
    Genera pares (X, y) con ventanas deslizantes para LSTM.

    Parámetros
    ----------
    data : np.ndarray
        Array 2D de shape (n_samples, n_features). Ya debe estar escalado.
    n_lags : int
        Longitud de la ventana (cuántos timesteps hacia atrás).
    target_col : int
        Índice de la columna que es el target.

    Retorna
    -------
    X : np.ndarray, shape (n_samples - n_lags, n_lags, n_features)
    y : np.ndarray, shape (n_samples - n_lags,)
    """
    X, y = [], []
    for i in range(len(data) - n_lags):
        X.append(data[i:i + n_lags])
        y.append(data[i + n_lags, target_col])
    return np.array(X), np.array(y)

# Test rápido de la función
ejemplo = np.arange(20).reshape(-1, 1)
X_test_fn, y_test_fn = create_sequences(ejemplo, n_lags=3)
print(f"Input de 20 valores → {X_test_fn.shape[0]} ventanas de shape {X_test_fn.shape[1:]}")
print(f"Primera ventana: {X_test_fn[0].flatten()} → predecir: {y_test_fn[0]}")
print(f"Segunda ventana: {X_test_fn[1].flatten()} → predecir: {y_test_fn[1]}")

In [ ]:
# Parámetros del modelado
N_LAGS = 12  # ventana de un año
print(f"🪟 Tamaño de ventana: {N_LAGS} meses (1 ciclo anual)")

# Convertir la serie a array 2D (necesario para StandardScaler)
train_vals = ts_train.values.reshape(-1, 1)
test_vals  = ts_test.values.reshape(-1, 1)

# Escalador ajustado SOLO con train
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_vals)

# Transformar test con el MISMO scaler
test_scaled = scaler.transform(test_vals)

print(f"\n📏 Train escalado: media={train_scaled.mean():.3f}, std={train_scaled.std():.3f}")
print(f"📏 Test escalado:  media={test_scaled.mean():.3f}, std={test_scaled.std():.3f}")
print(f"\n⚠️  El test escalado NO tiene media 0 ni std 1: es esperable, porque el scaler se ajustó con train.")

In [ ]:
# Crear ventanas de train
X_train, y_train = create_sequences(train_scaled, N_LAGS)

# Para el test, armamos las ventanas del forecast recursivo más adelante.
# Por ahora, guardamos la ÚLTIMA ventana del train (será la "semilla" del forecast).
ultima_ventana_train = train_scaled[-N_LAGS:]  # shape (12, 1)

print(f"📦 X_train shape: {X_train.shape}  → (samples={X_train.shape[0]}, timesteps={X_train.shape[1]}, features={X_train.shape[2]})")
print(f"📦 y_train shape: {y_train.shape}")
print(f"\n🌱 Semilla para forecast (última ventana del train): {ultima_ventana_train.flatten().round(3)}")

## 5. LSTM Simple — arquitectura univariada

### 5.1 ¿Qué es una LSTM, intuitivamente?

Una LSTM (Long Short-Term Memory) es una neurona con **memoria regulable**. Internamente tiene 3 "compuertas":

- **Compuerta de olvido**: decide qué parte de la memoria pasada descartar.
- **Compuerta de entrada**: decide qué información nueva incorporar.
- **Compuerta de salida**: decide qué parte de la memoria usar para producir la predicción.

En la práctica, el modelo **aprende solo** cuánta memoria conviene mantener para cada patrón. Para nuestra serie estacional, esperaríamos que retenga la información de hace 12 meses (ciclo anual) y olvide el ruido de corto plazo.

### 5.2 La arquitectura mínima

```
Input (12 timesteps, 1 feature)
   ↓
LSTM(32 unidades)     ← 32 "neuronas con memoria"
   ↓
Dense(1)              ← la predicción escalada del mes siguiente
```

**Por qué 32 unidades**: es un número conservador. Con más unidades (64, 128) el modelo tendría más capacidad pero también más riesgo de sobreajustar con solo 72 ventanas de entrenamiento.

**Por qué no agrego Dropout ni capas extra**: la regla con pocos datos es _empezar simple y complicarnos solo si hace falta_. Primero vemos qué hace esta red mínima.


In [ ]:
# Construcción del modelo
def build_lstm_simple(n_lags, n_features=1, units=32, lr=0.001):
    model = Sequential([
        Input(shape=(n_lags, n_features)),
        LSTM(units, activation='tanh'),
        Dense(1)
    ], name='LSTM_Simple')
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

# Resetear semillas (clave antes de cada modelo para reproducibilidad)
np.random.seed(SEED)
tf.random.set_seed(SEED)

lstm_simple = build_lstm_simple(N_LAGS, n_features=1)
lstm_simple.summary()

### 5.3 Entrenamiento

Usamos **EarlyStopping** con `patience=30`: si durante 30 épocas seguidas la pérdida de validación no mejora, el entrenamiento se detiene y restauramos los mejores pesos. Esto evita que el modelo sobreajuste y nos ahorra tiempo.

Usamos un 20% del train como validación interna. No tocamos el test — ese se reserva para evaluación final.


In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True,
    verbose=0
)

# Entrenamiento
history_simple = lstm_simple.fit(
    X_train, y_train,
    epochs=300,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    shuffle=False,   # ¡Importante! no mezclar ventanas temporales
    verbose=0
)

epochs_run = len(history_simple.history['loss'])
best_val_loss = min(history_simple.history['val_loss'])
print(f"⏱️  Entrenamiento finalizado en {epochs_run} épocas (patience=30)")
print(f"🎯 Mejor val_loss: {best_val_loss:.4f}")

In [ ]:
# Curvas de aprendizaje
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(history_simple.history['loss'], color=COLORS['accent'], label='Train loss', linewidth=1.8)
ax.plot(history_simple.history['val_loss'], color=COLORS['success'], label='Validation loss', linewidth=1.8)
ax.set_title('LSTM Simple — curvas de aprendizaje', fontsize=12, weight='bold')
ax.set_xlabel('Época'); ax.set_ylabel('MSE (escala estandarizada)')
ax.legend(); plt.tight_layout(); plt.show()

### 5.4 Forecast recursivo

Para predecir 12 meses hacia adelante **sin ver valores reales**, usamos una estrategia recursiva:

1. Tomamos la última ventana del train (meses ene-2017 a dic-2017).
2. Predecimos ene-2018.
3. **Incorporamos esa predicción** a la ventana (descartando el mes más antiguo).
4. Predecimos feb-2018 con la nueva ventana (que ya incluye ene-2018 predicho).
5. Repetimos 12 veces.

Este método simula un forecast real: el modelo no tiene acceso a ningún dato de 2018. La desventaja es que los errores se acumulan — si la predicción de enero es mala, todos los meses siguientes heredan ese error.

Es la misma lógica que usa ARIMA internamente, por lo que la comparación es limpia.


In [ ]:
def forecast_recursive_univariate(model, seed_window, n_steps, scaler):
    """
    Forecast recursivo para LSTM univariada.

    Parámetros
    ----------
    model : keras.Model
    seed_window : np.ndarray shape (n_lags, 1) — ya escalado
    n_steps : int — cuántos pasos predecir
    scaler : StandardScaler — para invertir la escala
    """
    window = seed_window.copy()
    predictions_scaled = []

    for _ in range(n_steps):
        x = window.reshape(1, *window.shape)  # (1, n_lags, 1)
        pred = model.predict(x, verbose=0)[0, 0]
        predictions_scaled.append(pred)

        # Desplazar ventana: sacar el más viejo, agregar la predicción al final
        window = np.roll(window, -1, axis=0)
        window[-1, 0] = pred

    # Invertir escala
    preds_array = np.array(predictions_scaled).reshape(-1, 1)
    return scaler.inverse_transform(preds_array).flatten()

# Pronosticar los 12 meses de 2018
forecast_simple = forecast_recursive_univariate(
    lstm_simple, ultima_ventana_train, n_steps=12, scaler=scaler
)

# Métricas
mae_simple  = mean_absolute_error(ts_test.values, forecast_simple)
rmse_simple = np.sqrt(mean_squared_error(ts_test.values, forecast_simple))
mape_simple = np.mean(np.abs((ts_test.values - forecast_simple) / ts_test.values)) * 100

print(f"📊 LSTM Simple (univariado):")
print(f"   MAE  = {mae_simple:.2f} GWh")
print(f"   RMSE = {rmse_simple:.2f} GWh")
print(f"   MAPE = {mape_simple:.2f}%")
print(f"\n   vs ARIMA: MAPE {mape_arima:.2f}%  →  Δ = {mape_simple - mape_arima:+.2f} puntos")

In [ ]:
# Visualización comparativa
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ts_train.index[-24:], ts_train.values[-24:],
        color=COLORS['accent'], linewidth=1.5, label='Train (últimos 24 meses)', alpha=0.7)
ax.plot(ts_test.index, ts_test.values,
        color=COLORS['primary'], linewidth=2.5, label='Real 2018', marker='o', markersize=6)
ax.plot(ts_test.index, arima_forecast.values,
        color=COLORS['gray'], linewidth=2, linestyle='--', marker='s', markersize=5,
        label=f'ARIMA — MAPE {mape_arima:.1f}%')
ax.plot(ts_test.index, forecast_simple,
        color=COLORS['success'], linewidth=2, linestyle='--', marker='^', markersize=5,
        label=f'LSTM Simple — MAPE {mape_simple:.1f}%')

ax.axvline(pd.Timestamp(SPLIT_DATE), color=COLORS['gray'], linestyle=':', alpha=0.5)
ax.set_title('Forecast 2018: ARIMA vs LSTM Simple', fontsize=13, weight='bold')
ax.set_xlabel('Mes'); ax.set_ylabel('GWh')
ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

## 6. LSTM Multivariada — ¿ayuda el contexto externo?

La LSTM Simple usa **solo la historia de la propia serie** (lo que se llama _modelo autoregresivo puro_). Pero tenemos más información disponible que podría ayudar:

### Features que agregamos

1. **`demanda_MEM_GWh`** — la demanda total del sistema eléctrico. Hipótesis: si la demanda crece, hay más incentivo a conectar renovables (que son el último eslabón en la merit-order de despacho).

2. **Encoding cíclico del mes** (`mes_sin`, `mes_cos`). Aquí hay una sutileza importante:

### ¿Por qué no usar "1, 2, 3, ..., 12" como el número del mes?

Porque ese encoding **miente sobre la distancia**: el modelo vería que diciembre (12) y enero (1) están _a 11 unidades_ de distancia, cuando en realidad están _a 1 mes_ (son vecinos en el calendario).

La solución es proyectar el mes sobre un círculo unitario:

```
mes_sin = sin(2π · mes / 12)
mes_cos = cos(2π · mes / 12)
```

Ahora diciembre y enero son **verdaderos vecinos en el espacio de features**. Este truco se llama **encoding cíclico** y es estándar en series temporales.


In [ ]:
# Feature engineering
df_multi = df[['mes', 'gen_renovable_total_GWh', 'demanda_MEM_GWh']].copy()
df_multi['month_num'] = df_multi['mes'].dt.month
df_multi['mes_sin']   = np.sin(2 * np.pi * df_multi['month_num'] / 12)
df_multi['mes_cos']   = np.cos(2 * np.pi * df_multi['month_num'] / 12)

# Visualizar el encoding cíclico
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Círculo
axes[0].scatter(df_multi['mes_cos'].head(12), df_multi['mes_sin'].head(12),
                c=df_multi['month_num'].head(12), cmap='twilight', s=120, edgecolor='black')
for i in range(12):
    axes[0].annotate(str(i+1), (df_multi['mes_cos'].iloc[i], df_multi['mes_sin'].iloc[i]),
                     ha='center', va='center', fontsize=9, weight='bold')
axes[0].set_title('Meses proyectados en círculo unitario', fontsize=11, weight='bold')
axes[0].set_xlabel('mes_cos'); axes[0].set_ylabel('mes_sin')
axes[0].axhline(0, color='gray', alpha=0.3); axes[0].axvline(0, color='gray', alpha=0.3)
axes[0].set_aspect('equal')

# Serie temporal con demanda
axes[1].plot(df_multi['mes'], df_multi['demanda_MEM_GWh'], color=COLORS['warning'], linewidth=1.5)
axes[1].set_title('Demanda eléctrica (MEM) mensual', fontsize=11, weight='bold')
axes[1].set_xlabel('Mes'); axes[1].set_ylabel('GWh')
plt.tight_layout(); plt.show()

print("✅ Feature engineering listo: gen_renovable + demanda + mes_sin + mes_cos")

In [ ]:
# Separar features y target
FEATURES = ['gen_renovable_total_GWh', 'demanda_MEM_GWh', 'mes_sin', 'mes_cos']
TARGET_IDX = 0  # gen_renovable está en la primera columna

data_multi = df_multi[FEATURES].values
print(f"📊 Shape total: {data_multi.shape}  (96 meses × 4 features)")

# Split temporal
train_multi = data_multi[:len(ts_train)]    # 84 meses
test_multi  = data_multi[len(ts_train):]    # 12 meses

# Escalado — solo las columnas continuas (no las cíclicas, ya están en [-1, 1])
# Usamos DOS scalers: uno para el target, otro para la demanda
scaler_target_m = StandardScaler()
scaler_demanda  = StandardScaler()

train_multi_scaled = train_multi.copy()
train_multi_scaled[:, [0]] = scaler_target_m.fit_transform(train_multi[:, [0]])
train_multi_scaled[:, [1]] = scaler_demanda.fit_transform(train_multi[:, [1]])

test_multi_scaled = test_multi.copy()
test_multi_scaled[:, [0]] = scaler_target_m.transform(test_multi[:, [0]])
test_multi_scaled[:, [1]] = scaler_demanda.transform(test_multi[:, [1]])

print(f"\n📏 Train escalado — primer fila: {train_multi_scaled[0].round(3)}")
print(f"📏 Train escalado — media por col: {train_multi_scaled.mean(axis=0).round(3)}")

In [ ]:
# Crear ventanas multivariadas
X_train_m, y_train_m = create_sequences(train_multi_scaled, N_LAGS, target_col=TARGET_IDX)

# Semilla del forecast: última ventana del train
ultima_ventana_multi = train_multi_scaled[-N_LAGS:]

print(f"📦 X_train_m shape: {X_train_m.shape}  → (samples, timesteps, features)")
print(f"📦 y_train_m shape: {y_train_m.shape}")
print(f"🌱 Semilla shape: {ultima_ventana_multi.shape}")

### 6.1 Arquitectura — misma filosofía: simple

Mantenemos la arquitectura lo más parecida posible a la LSTM Simple para que la comparación sea **ceteris paribus**: **la única diferencia es el input**, no la red. Así podemos atribuir cualquier mejora (o empeoramiento) al aporte de las features externas, no a la capacidad del modelo.

```
Input (12 timesteps, 4 features)
   ↓
LSTM(32 unidades)
   ↓
Dense(1)
```

Única diferencia: el input ahora tiene **4 features por timestep** en lugar de 1.


In [ ]:
# Modelo multivariado — misma arquitectura, distinto input
np.random.seed(SEED)
tf.random.set_seed(SEED)

lstm_multi = build_lstm_simple(N_LAGS, n_features=len(FEATURES))
lstm_multi._name = 'LSTM_Multivariada'
lstm_multi.summary()

In [ ]:
# Entrenamiento
history_multi = lstm_multi.fit(
    X_train_m, y_train_m,
    epochs=300,
    batch_size=8,
    validation_split=0.2,
    callbacks=[EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=0)],
    shuffle=False,
    verbose=0
)

epochs_run_m = len(history_multi.history['loss'])
print(f"⏱️  Entrenamiento finalizado en {epochs_run_m} épocas")
print(f"🎯 Mejor val_loss: {min(history_multi.history['val_loss']):.4f}")

In [ ]:
# Curvas de aprendizaje comparadas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_simple.history['loss'], color=COLORS['accent'], label='Train', linewidth=1.8)
axes[0].plot(history_simple.history['val_loss'], color=COLORS['success'], label='Val', linewidth=1.8)
axes[0].set_title(f'LSTM Simple ({epochs_run} épocas)', fontsize=11, weight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('MSE'); axes[0].legend()

axes[1].plot(history_multi.history['loss'], color=COLORS['accent'], label='Train', linewidth=1.8)
axes[1].plot(history_multi.history['val_loss'], color=COLORS['warning'], label='Val', linewidth=1.8)
axes[1].set_title(f'LSTM Multivariada ({epochs_run_m} épocas)', fontsize=11, weight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('MSE'); axes[1].legend()

plt.tight_layout(); plt.show()

### 6.2 Forecast recursivo multivariado

Para el forecast recursivo multivariado, hay un punto delicado: **cada mes futuro necesita sus features exógenas**.

- `mes_sin`, `mes_cos` → sin problema, son determinísticos por el calendario (ene-2018 tiene los mismos valores que ene de cualquier año).
- `demanda_MEM` → **no la conocemos a priori**. Para este ejercicio **usamos la demanda real de 2018**, lo que equivale a asumir "demanda conocida" (un escenario común en planificación energética: las distribuidoras proyectan su demanda).

Es un benchmark honesto: le damos a la LSTM la información contextual y vemos si la sabe aprovechar.


In [ ]:
def forecast_recursive_multivariate(model, seed_window, future_exog_scaled, n_steps,
                                      scaler_target, target_idx=0):
    """
    Forecast recursivo multivariado.

    Parámetros
    ----------
    seed_window : shape (n_lags, n_features) escalado
    future_exog_scaled : shape (n_steps, n_features) — datos del test ya escalados
                        (usamos las features exógenas reales; el target lo predecimos)
    """
    window = seed_window.copy()
    predictions_scaled = []

    for step in range(n_steps):
        x = window.reshape(1, *window.shape)
        pred = model.predict(x, verbose=0)[0, 0]
        predictions_scaled.append(pred)

        # Nueva fila: predicción + features exógenas REALES del mes a predecir
        new_row = future_exog_scaled[step].copy()
        new_row[target_idx] = pred  # reemplazar el target por la predicción

        # Desplazar ventana
        window = np.roll(window, -1, axis=0)
        window[-1] = new_row

    # Invertir escala del target
    preds_array = np.array(predictions_scaled).reshape(-1, 1)
    return scaler_target.inverse_transform(preds_array).flatten()

# Forecast
forecast_multi = forecast_recursive_multivariate(
    lstm_multi,
    ultima_ventana_multi,
    test_multi_scaled,   # incluye features exógenas reales
    n_steps=12,
    scaler_target=scaler_target_m,
    target_idx=TARGET_IDX
)

# Métricas
mae_multi  = mean_absolute_error(ts_test.values, forecast_multi)
rmse_multi = np.sqrt(mean_squared_error(ts_test.values, forecast_multi))
mape_multi = np.mean(np.abs((ts_test.values - forecast_multi) / ts_test.values)) * 100

print(f"📊 LSTM Multivariada:")
print(f"   MAE  = {mae_multi:.2f} GWh")
print(f"   RMSE = {rmse_multi:.2f} GWh")
print(f"   MAPE = {mape_multi:.2f}%")
print(f"\n   vs ARIMA:        Δ = {mape_multi - mape_arima:+.2f} puntos")
print(f"   vs LSTM Simple:  Δ = {mape_multi - mape_simple:+.2f} puntos")

## 7. Comparación final y storytelling

Llegó el momento de poner los tres modelos en la misma balanza.


In [ ]:
# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo': ['ARIMA(1,1,1)', 'LSTM Simple (univariada)', 'LSTM Multivariada'],
    'Tipo': ['Estadístico lineal', 'Deep Learning', 'Deep Learning + contexto'],
    'MAE (GWh)':  [mae_arima, mae_simple, mae_multi],
    'RMSE (GWh)': [rmse_arima, rmse_simple, rmse_multi],
    'MAPE (%)':   [mape_arima, mape_simple, mape_multi],
})
resultados = resultados.round(2)

# Marcar el mejor
best_idx = resultados['MAPE (%)'].idxmin()
print(f"🏆 Mejor modelo (menor MAPE): {resultados.iloc[best_idx]['Modelo']}")
print()
resultados

In [ ]:
# Visualización comparativa final
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [2, 1]})

# Panel 1: forecasts
ax = axes[0]
ax.plot(ts_train.index[-18:], ts_train.values[-18:],
        color=COLORS['accent'], linewidth=1.5, alpha=0.6, label='Train (últimos 18 meses)')
ax.plot(ts_test.index, ts_test.values,
        color=COLORS['primary'], linewidth=2.8, marker='o', markersize=7, label='Real 2018')
ax.plot(ts_test.index, arima_forecast.values,
        color=COLORS['gray'], linewidth=2, linestyle='--', marker='s', markersize=5,
        label=f'ARIMA — MAPE {mape_arima:.1f}%')
ax.plot(ts_test.index, forecast_simple,
        color=COLORS['success'], linewidth=2, linestyle='--', marker='^', markersize=5,
        label=f'LSTM Simple — MAPE {mape_simple:.1f}%')
ax.plot(ts_test.index, forecast_multi,
        color=COLORS['warning'], linewidth=2, linestyle='--', marker='D', markersize=5,
        label=f'LSTM Multivariada — MAPE {mape_multi:.1f}%')
ax.axvline(pd.Timestamp(SPLIT_DATE), color=COLORS['gray'], linestyle=':', alpha=0.5)
ax.set_title('Forecast 2018 — Los tres modelos lado a lado', fontsize=13, weight='bold')
ax.set_xlabel(''); ax.set_ylabel('GWh'); ax.legend(loc='upper left', fontsize=9)

# Panel 2: residuos
ax = axes[1]
residuos = pd.DataFrame({
    'mes': ts_test.index,
    'ARIMA': ts_test.values - arima_forecast.values,
    'LSTM Simple': ts_test.values - forecast_simple,
    'LSTM Multivariada': ts_test.values - forecast_multi,
}).set_index('mes')

x_pos = np.arange(len(residuos))
width = 0.27

ax.bar(x_pos - width, residuos['ARIMA'], width, color=COLORS['gray'], label='ARIMA')
ax.bar(x_pos,         residuos['LSTM Simple'], width, color=COLORS['success'], label='LSTM Simple')
ax.bar(x_pos + width, residuos['LSTM Multivariada'], width, color=COLORS['warning'], label='LSTM Multivariada')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels([d.strftime('%b') for d in residuos.index], rotation=0)
ax.set_title('Residuos (real − predicho) — signo positivo = el modelo subestimó',
             fontsize=12, weight='bold')
ax.set_ylabel('Residuo (GWh)'); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

### 📝 Interpretación de los resultados

**Lectura general de los residuos:** los tres modelos tienden a **subestimar** 2018 (residuos positivos casi todo el año). Esto es la firma del cambio estructural: ninguno de los modelos vio nunca un régimen como el de 2018 durante el train, así que todos se quedan cortos — pero en distintas magnitudes.

### 🎯 Conclusiones del Paso 4.2

**Lo que aprendimos sobre LSTM:**

### 1️⃣ Las LSTM superan claramente al baseline ARIMA
Ambas arquitecturas LSTM (Simple y Multivariada) le ganan a ARIMA. Este es el **primer momento del Desafío donde Deep Learning supera claramente a un baseline estadístico** — y tiene mucho sentido: la memoria de secuencias es exactamente la ventaja estructural que ARIMA no tiene. Cuando hay un cambio estructural no lineal (como el boom RenovAr de 2018), la flexibilidad de la red neuronal se nota.

### 2️⃣ El hallazgo más interesante: **la LSTM Simple le gana a la Multivariada**

A primera vista contraintuitivo: ¿cómo puede ser que _agregar información_ empeore el modelo? La respuesta está en un trade-off fundamental del Machine Learning: **más capacidad ≠ mejor generalización cuando los datos son pocos**.

| Aspecto | LSTM Simple | LSTM Multivariada |
|---|---|---|
| Features de entrada | 1 | 4 |
| Parámetros del modelo (~) | ~4.4k | ~4.8k |
| Ejemplos de entrenamiento | 72 | 72 |
| Ratio parámetros/datos | 61:1 | 67:1 |

La multivariada tiene **más capacidad de memorizar el train** sin generalizar mejor al test. Con 72 ventanas de entrenamiento y 4 features, el espacio de combinaciones a aprender es mucho más grande y el modelo tiende a **sobreajustar**.

Este resultado ilustra una **lección clásica y profunda**: el **principio de parsimonia** (o _Navaja de Occam_ aplicada a ML): entre dos modelos con performance similar, **preferí el más simple** — generaliza mejor y es más robusto.

### 3️⃣ El encoding cíclico sigue siendo una técnica valiosa

Aunque la multivariada no ganó aquí, el encoding cíclico del mes (sin/cos) sigue siendo una técnica estándar y poderosa en series temporales. Con más datos de entrenamiento, la multivariada probablemente ganaría.

### 4️⃣ Comparación honesta con ARIMA

ARIMA es un modelo **con 3 parámetros**. La LSTM Simple tiene **miles**. Que gane por 6 puntos de MAPE es bueno, pero la diferencia no es abismal. En un escenario con pocos datos y cambios estructurales fuertes, ARIMA sigue siendo un baseline sorprendentemente competitivo — y la simplicidad tiene un valor: ARIMA es interpretable, la LSTM es una caja negra.

**Limitaciones honestas:**

- **Datos escasos:** 72 ventanas de entrenamiento es poco. Una validación cruzada temporal rigurosa podría mostrar varianza alta entre folds.
- **Una sola corrida:** con otra semilla los resultados podrían variar. Lo ideal sería promediar ≥5 corridas.
- **Forecast recursivo:** los errores se propagan. Cuanto más lejos predicimos, peor.
- **Benchmark con covariables conocidas:** la LSTM Multivariada usa la demanda real de 2018. En un escenario 100% predictivo habría que pronosticar la demanda también.

### 🔄 ¿Qué sigue?

- **Paso 4.3:** Fine tuning — probar diferentes tamaños de ventana (6, 12, 18, 24), más unidades, regularización (Dropout, L2). ¿Podemos cerrar aún más el gap?
- **Paso 4.4:** Gran comparación ML vs Deep Learning — poner todo sobre la mesa.
- **Paso 4.5:** Storytelling final del proyecto.

> 💡 **Takeaway del Paso 4.2:** Deep Learning aporta valor real en series temporales con dinámica no lineal — pero la arquitectura debe ser proporcional a la cantidad de datos. Con series cortas, _"menos es más"_ no es una frase hecha: es un principio estadístico.
